In [5]:
def print_translations():
    with open("./en-zh.en-filtered.en.subword.test.desubword", 'r', encoding='utf-8') as en_file, \
         open("./zh.translated.desubword", 'r', encoding='utf-8') as zh_salient_file, \
         open("./zh.base.translated.desubword", 'r', encoding='utf-8') as zh_base_file, \
         open("./en-zh.zh-filtered.zh.subword.test.desubword", encoding="utf-8") as zh_correct_file:
        
        for en_line, zh_salient_line, zh_base_line, zh_correct_line in zip(en_file, zh_salient_file, zh_base_file, zh_correct_file):
            en_line = en_line.strip()
            zh_salient_line = zh_salient_line.strip()
            zh_base_line = zh_base_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            print(f"English:            {en_line}")
            print(f"Chinese (salient):  {zh_salient_line}")
            print(f"Chinese (base):     {zh_base_line}")
            print(f"Chinese (correct):  {zh_correct_line}")
            print("-" * 50)

print_translations()


English:            Before I know it, she has skiddled across the parking lot and in between the cars, and people behind me, with that kind of "I'm coming." Italian hand signals follow.
Chinese (salient):  在我还没有意识到之前, 她已经跨越了停车场和车间, 在我身后的人, 带着这样的“我要过来”,意大利式的手势跟随了。
Chinese (base):     如果我把乌干达分解开,在乌干达有很大的不同。
Chinese (correct):  ▁我还没回过神儿,她已轻松穿行于停车场的汽车中, 身后的人们看我的眼神, 充满了惊羡。哇啊——哇啊——
--------------------------------------------------
English:            If I split Uganda, there's quite a difference within Uganda.
Chinese (salient):  如果我把乌干达分割开,乌干达境内也有很大的差别。
Chinese (base):     我不知道, 这个项目有很多可能性, 我鼓励你们所有人, 记录下你生活的一个小片段, 所以你永远忘不了那天,你真的活着。
Chinese (correct):  ▁如果把乌干达分解开 可以看到内部的明显差异
--------------------------------------------------
English:            I look at this photo, and he seems really interested in what's going on with that button, but it doesn't seem like he is really that interested in crossing the street.
Chinese (salient):  我看着这张照片, 他似乎对那个按钮上有什么特别感兴趣, 但是看起来他对过马路 没那么感兴趣。
Chinese (base):

In [ ]:
import nltk
from nltk.translate.meteor_score import meteor_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.metrics.distance import edit_distance
from collections import defaultdict
import pandas as pd
import jieba

nltk.download('punkt')

In [46]:
def analyze_translations_meteor(en_path, zh_trans_path, zh_correct_path):
    """
    Analyzes translations using METEOR and edit distance metrics.

    Parameters:
    - en_path: Path to the English source sentences.
    - zh_trans_path: Path to the translated Chinese sentences.
    - zh_correct_path: Path to the reference Chinese translations.

    Returns:
    - df_sentences: DataFrame containing sentence-level analysis.
    - df_mismatches: DataFrame containing word mismatch counts.
    """
    sentence_results = []  # To store sentence-level metrics
    word_mismatches = defaultdict(int)  # To tally missing/mistranslated words
    smoothing = SmoothingFunction().method1  # Smoothing for sentence-level BLEU

    with open(en_path, 'r', encoding='utf-8') as en_file, \
         open(zh_trans_path, 'r', encoding='utf-8') as zh_file, \
         open(zh_correct_path, 'r', encoding='utf-8') as zh_correct_file:
        
        for idx, (en_line, zh_line, zh_correct_line) in enumerate(zip(en_file, zh_file, zh_correct_file)):
            en_line = en_line.strip()
            zh_line = zh_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            # Tokenize the Chinese sentences
            zh_tokens = list(jieba.cut(zh_line))
            zh_correct_tokens = list(jieba.cut(zh_correct_line))

            # Compute the sentence-level scores.
            meteor = meteor_score([zh_correct_tokens], zh_tokens)
            bleu = sentence_bleu([zh_correct_tokens], zh_tokens, smoothing_function=smoothing)
            dist = edit_distance(zh_line, zh_correct_line)
            
            sentence_results.append({
                'English': en_line,
                'Chinese (model)': zh_line,
                'Chinese (correct)': zh_correct_line,
                'METEOR': meteor,
                'BLEU': bleu,
                'EditDistance': dist
            })
            
            # Tally missing words from the reference that do not appear in the model output
            for word in zh_correct_tokens:
                if word not in zh_tokens:
                    word_mismatches[word] += 1
                    
    df_sentences = pd.DataFrame(sentence_results)
    df_mismatches = pd.DataFrame(list(word_mismatches.items()), columns=['Word', 'MismatchCount'])
    df_mismatches.sort_values(by='MismatchCount', ascending=False, inplace=True)
    
    return df_sentences, df_mismatches


In [47]:
en_path = "./en-zh.en-filtered.en.subword.test.desubword"
zh_trans_path = "./zh.translated.desubword"
zh_correct_path = "./en-zh.zh-filtered.zh.subword.test.desubword"

df_sentences, df_mismatches = analyze_translations_meteor(en_path, zh_trans_path, zh_correct_path)

### Sort by METEOR (asc), edit distance (desc)

Edit distance is used as well to highlight sentences that are semantically different

In [71]:
df_sentences.sort_values(by=['METEOR', 'EditDistance'], ascending=[True, False]).head(10)

,English,Chinese (model),Chinese (correct),METEOR,BLEU,EditDistance
587,"You make a PowerPoint, you know?","你制作了Powerpoint,你们知道吗?",我们需要Power Point,0.0,0.0,13
1539,To her girlfriends she said that.,女友说她这么说。,”我知道我自己在做什么“,0.0,0.0,12
361,That's an amazing thing.,这是件令人惊叹的事。,很神奇吧!,0.0,0.0,10
741,It's transformational.,它是一种转型。,这将带来伟大的变革.,0.0,0.0,10
1226,You can see this one.,你可以看到这个。,"看着这怪兽,从左到右",0.0,0.0,10
1589,This is Paldin.,这是Palindin。,"这位是宝丁,",0.0,0.0,10
1219,The textile industry is incredibly mobile.,"纺织业非常灵活,",纺织品行业极具移动性。,0.0,0.0,8
184,Audience: Yes! Yeah!,观众:没错!,"是的,观众们。",0.0,0.0,7
417,So the upshot was this.,截图就是这个,这是结果。,0.0,0.0,6
740,Us.,美,是我们自己。,0.0,0.0,6


### Sort sentences based on input length

In [67]:
df_sentences.sort_values(by='English', key=lambda x: x.str.len()).head(10)

,English,Chinese (model),Chinese (correct),METEOR,BLEU,EditDistance
740,Us.,美,是我们自己。,0.000000,0.000000,6
563,7.5.,7.5次。,7.5次。,0.981481,0.562341,0
591,Mmm.,嗯,嗯,0.500000,0.177828,0
855,Thanks.,谢谢。,谢谢。,0.937500,0.316228,0
552,E: Yar.,爱因斯坦:,爱因斯坦:是。,0.493421,0.116334,2
1414,Me too.,我也是。,我也是。,0.992188,1.000000,0
1350,Stop it.,停止吧,停下吧,0.250000,0.149535,1
1082,Namaste.,马斯特。,谢谢。,0.250000,0.149535,3
1746,Yes? OK.,是吗?好的。,愿意?好的,0.701058,0.202052,3
578,Owl. Owl.,猫头鹰。猫头鹰。,猫头鹰。猫头鹰。,0.992188,1.000000,0


In [74]:
print("\nMost Frequently Mistranslated (Missing) Words:")
df_mismatches.head()


Most Frequently Mistranslated (Missing) Words:


,Word,MismatchCount
90,的,209
26,是,183
25,了,138
18,",",126
64,在,125


### Result findings
- Mistranslated Entity
- Not capturing the correct sense
  - e.g And in summer, here, killer wasps.
  - e.g 法律人员 vs 合法的人 in "Humans and legal persons are not synonymous."
- Cannot translate well if 1 token for input (?)
  - e.g Us.;美;	是我们自己。
- Captured sentences literally (without considering enough context) 
  - Not good because the model is suppose to look at the entire corpus (?)
  - e.g It was all about going for the center.
- Wrong inversion
  - e.g Nobody wants to buy a mini well when they buy a car.
- Inaccuracy in capturing the correct sense
  - e.g It's a mind setting.
- Wrong intensity of adjective
  - e.g I was less exotic in this Whitopia.

## Approach to Find Polysemous Sentences

### Approach 1
- before adding salient words
- use a python dictionary of polysemous words

In [30]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

# Download necessary NLTK resources silently
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)  # Required for Chinese translations in WordNet
nltk.download("averaged_perceptron_tagger_eng", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("wordnet_ic", quiet=True)

# File paths
en_file_path = "./en-zh.en-filtered.en.subword.test.desubword"
zh_file_path = "./zh.translated.desubword"
zh_correct_file_path = "./en-zh.zh-filtered.zh.subword.test.desubword"

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

# Common polysemous words (expandable)
polysemous_words = {"bank", "light", "touch", "lead", "bark", "current", "rock", "date", "crane", "seal"}

def get_wordnet_pos(word):
    """Convert POS tag to a format WordNet understands."""
    tag_dict = {"J": wn.ADJ, "N": wn.NOUN, "V": wn.VERB, "R": wn.ADV}
    
    try:
        pos = pos_tag([word])[0][1][0].upper()  # Get first letter of POS tag
        return tag_dict.get(pos, wn.NOUN)  # Default to NOUN if not found
    except Exception as e:
        print(f"POS tagging error for word: {word} -> {e}")
        return wn.NOUN

def lemmatize_word(word):
    """Lemmatize a word using its POS tag."""
    return lemmatizer.lemmatize(word, get_wordnet_pos(word))

def is_polysemous(word):
    """Check if a word has multiple meanings in WordNet."""
    return len(wn.synsets(word)) > 1

def get_possible_translations(word):
    """Get possible Chinese translations from WordNet."""
    translations = set()
    for synset in wn.synsets(word):
        for lemma in synset.lemmas("cmn"):  # 'cmn' is the code for Mandarin Chinese
            translations.add(lemma.name())
    return translations

def detect_polysemy_errors():
    with open(en_file_path, 'r', encoding='utf-8') as en_file, \
         open(zh_file_path, 'r', encoding='utf-8') as zh_file, \
         open(zh_correct_file_path, 'r', encoding='utf-8') as zh_correct_file:
        
        for en_line, zh_line, zh_correct_line in zip(en_file, zh_file, zh_correct_file):
            en_line = en_line.strip()
            zh_line = zh_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            words = set(word_tokenize(en_line.lower()))   # Tokenize sentence
            lemmatized_words = {lemmatize_word(word) for word in words}  # Lemmatize each word
            
            polysemous_in_sentence = lemmatized_words.intersection(polysemous_words)  # Find polysemous words
            
            for word in polysemous_in_sentence:
                possible_translations = get_possible_translations(word)

                if possible_translations:
                    incorrect_mistranslated = any(trans in zh_line for trans in possible_translations)
                    correct_translated = any(trans in zh_correct_line for trans in possible_translations)

                    if incorrect_mistranslated and not correct_translated:
                        print(f"Possible polysemy-based error detected:")
                        print(f"- English: {en_line}")
                        print(f"- Incorrect Chinese: {zh_line}")
                        print(f"- Correct Chinese: {zh_correct_line}")
                        print(f"- Word causing issue: {word}")
                        print(f"- Possible translations: {possible_translations}")
                        print("-" * 50)

detect_polysemy_errors()


Possible polysemy-based error detected:
- English: And we were particularly touched by the flowers and we were curious as to how the flowers got there.
- Incorrect Chinese: 我们特别受到花朵的触动, 我们很好奇花是怎么摆到那里的。
- Correct Chinese: 特别是那些花,它们尤其让我们感动。 我们很好奇,那些花是怎么摆到那里的呢?”
- Word causing issue: touch
- Possible translations: {'影响', '伸出', '知觉', '涉及', '暗指', '间接提到', '联络', '轻微的侵害', '比得上', '接触', '达到', '少许', '碰到', '身体接触', '摸', '有关', '使接触', '损害', '使相碰', '感觉', '触觉', '碰', '关系到', '触摸', '少量', '作用', '触', '会晤', '触及'}
--------------------------------------------------
Possible polysemy-based error detected:
- English: But on the other hand, we have 14 billion of these: light bulbs, light.
- Incorrect Chinese: 但另一方面, 我们有140亿种灯泡,光亮。
- Correct Chinese: 另一方面, 我们有一百四十亿个 灯泡.
- Word causing issue: light
- Possible translations: {'轻松+地', '容易+地', '点火器', '不足+的', '缺乏+的', '下马', '启发', '明亮+的', '打火机', '昏厥+的', '眩晕+的', '欠缺+的', '毫无约束+的', '放荡+的', '启示', '启蒙', '颜色浅+的', '无意义+的', '点起', '微不足道+的', '点火', '无价值+的', '光源', '光度', '虚弱+的', '轻便+地

### Approach 2
- before adding salient words
- through lemmatization + NLTK's WordNet

In [29]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

# Ensure NLTK resources are downloaded silently
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("punkt", quiet=True)

# File paths
en_file_path = "./en-zh.en-filtered.en.subword.test.desubword"
zh_file_path = "./zh.translated.desubword"
zh_correct_file_path = "./en-zh.zh-filtered.zh.subword.test.desubword"

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def get_polysemous_words(min_synsets=2, word_limit=500):
    """
    Extracts polysemous words from WordNet.
    A word is considered polysemous if it has at least `min_synsets` meanings.
    """
    polysemous_words = set()
    for synset in wn.all_synsets():
        for lemma in synset.lemmas():
            word = lemma.name().replace('_', ' ')  # Convert WordNet format (e.g., 'rock_n_1' → 'rock')
            if len(wn.synsets(word)) >= min_synsets:  # Ensure multiple meanings
                polysemous_words.add(word)
            if len(polysemous_words) >= word_limit:
                return polysemous_words
    return polysemous_words

# Dynamically generated polysemous words
polysemous_words = get_polysemous_words()

def get_wordnet_pos(word):
    """Convert POS tag to a format WordNet understands."""
    tag_dict = {"J": wn.ADJ, "N": wn.NOUN, "V": wn.VERB, "R": wn.ADV}
    
    try:
        pos = pos_tag([word])[0][1][0].upper()  # Get first letter of POS tag
        return tag_dict.get(pos, wn.NOUN)  # Default to NOUN if not found
    except Exception as e:
        return wn.NOUN  # Fallback to NOUN

def lemmatize_word(word):
    """Lemmatize a word using its POS tag."""
    return lemmatizer.lemmatize(word, get_wordnet_pos(word))

def get_possible_translations(word):
    """Get possible Chinese translations from WordNet."""
    translations = set()
    for synset in wn.synsets(word):
        for lemma in synset.lemmas("cmn"):  # 'cmn' is the code for Mandarin Chinese
            translations.add(lemma.name())
    return translations

def detect_polysemy_errors():
    with open(en_file_path, 'r', encoding='utf-8') as en_file, \
         open(zh_file_path, 'r', encoding='utf-8') as zh_file, \
         open(zh_correct_file_path, 'r', encoding='utf-8') as zh_correct_file:
        
        for en_line, zh_line, zh_correct_line in zip(en_file, zh_file, zh_correct_file):
            en_line = en_line.strip()
            zh_line = zh_line.strip()
            zh_correct_line = zh_correct_line.strip()
            
            words = set(word_tokenize(en_line.lower()))  # Tokenize and lowercase sentence
            lemmatized_words = {lemmatize_word(word) for word in words}  # Lemmatize words
            
            polysemous_in_sentence = lemmatized_words.intersection(polysemous_words)  # Find polysemous words
            
            for word in polysemous_in_sentence:
                possible_translations = get_possible_translations(word)

                if possible_translations:
                    incorrect_mistranslated = any(trans in zh_line for trans in possible_translations)
                    correct_translated = any(trans in zh_correct_line for trans in possible_translations)

                    if incorrect_mistranslated and not correct_translated:
                        print(f"Possible polysemy-based error detected:")
                        print(f"- English: {en_line}")
                        print(f"- Incorrect Chinese: {zh_line}")
                        print(f"- Correct Chinese: {zh_correct_line}")
                        print(f"- Word causing issue: {word}")
                        print(f"- Possible translations: {possible_translations}")
                        print("-" * 50)

detect_polysemy_errors()


Possible polysemy-based error detected:
- English: I've been wondering for a long time, since I've been thinking about memes a lot, is there a difference between the memes that we copy -- the words we speak to each other, the gestures we copy, the human things -- and all these technological things around us?
- Incorrect Chinese: 我很久以来在思考迷因是什么, 自从我思考迷因的时候, 我们所复制的迷因之间的差别-- 我们互相说的话, 我们模仿的手势,人类的手势, 以及我们身边所有科技的事物?
- Correct Chinese: 很长一段时间我都在思考, 从我常常思考迷因开始, 我们所复制的迷因之间的差别-- 我们互相说的话, 我们模仿的手势,人类之间的那些事-- 以及所有这些我们周围的技术?
- Word causing issue: long
- Possible translations: {'长+的', '渴望', '有记性+的', '长时间+的', '长久+的', '记性强+的', '冒险的', '能记住+的', '记性好+的', '长期+的', '较长期间+的', '记忆力强+的', '冗长+的', '相对高+的', '久'}
--------------------------------------------------
Possible polysemy-based error detected:
- English: There are very important people, business and land assets in Detroit, and there are real opportunities there.
- Incorrect Chinese: 底特律有很多很重要的人员, 商业和土地资产, 也有真正的机会。
- Correct Chinese: 底特律仍有着十分重要的人群、 产业以及土地, 而

### Comparing before and after adding salient words

In [ ]:
import nltk
from nltk.corpus import wordnet as wn
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag, word_tokenize

# Ensure NLTK resources are downloaded silently
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("punkt", quiet=True)

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def get_polysemous_words(min_synsets=2, word_limit=500):
    """
    Extracts polysemous words from WordNet.
    A word is considered polysemous if it has at least `min_synsets` meanings.
    """
    polysemous_words = set()
    for synset in wn.all_synsets():
        for lemma in synset.lemmas():
            word = lemma.name().replace('_', ' ')  # Convert WordNet format (e.g., 'rock_n_1' → 'rock')
            if len(wn.synsets(word)) >= min_synsets:  # Ensure multiple meanings
                polysemous_words.add(word)
            if len(polysemous_words) >= word_limit:
                return polysemous_words
    return polysemous_words

# Dynamically generated polysemous words
polysemous_words = get_polysemous_words()

def get_wordnet_pos(word):
    """Convert POS tag to a format WordNet understands."""
    tag_dict = {"J": wn.ADJ, "N": wn.NOUN, "V": wn.VERB, "R": wn.ADV}
    
    try:
        pos = pos_tag([word])[0][1][0].upper()  # Get first letter of POS tag
        return tag_dict.get(pos, wn.NOUN)  # Default to NOUN if not found
    except Exception as e:
        return wn.NOUN  # Fallback to NOUN

def lemmatize_word(word):
    """Lemmatize a word using its POS tag."""
    return lemmatizer.lemmatize(word, get_wordnet_pos(word))

def get_possible_translations(word):
    """Get possible Chinese translations from WordNet."""
    translations = set()
    for synset in wn.synsets(word):
        for lemma in synset.lemmas("cmn"):  # 'cmn' is the code for Mandarin Chinese
            translations.add(lemma.name())
    return translations

def detect_polysemy_errors(en_file_path, zh_file_path, zh_correct_file_path):
    errors_detected = []
    with open(en_file_path, 'r', encoding='utf-8') as en_file, \
         open(zh_file_path, 'r', encoding='utf-8') as zh_file, \
         open(zh_correct_file_path, 'r', encoding='utf-8') as zh_correct_file:
        
        for en_line, zh_line, zh_correct_line in zip(en_file, zh_file, zh_correct_file):
            en_line, zh_line, zh_correct_line = en_line.strip(), zh_line.strip(), zh_correct_line.strip()
            words = set(word_tokenize(en_line.lower()))
            lemmatized_words = {lemmatize_word(word) for word in words}
            polysemous_in_sentence = lemmatized_words.intersection(polysemous_words)
            
            for word in polysemous_in_sentence:
                possible_translations = get_possible_translations(word)
                if possible_translations:
                    incorrect_mistranslated = any(trans in zh_line for trans in possible_translations)
                    correct_translated = any(trans in zh_correct_line for trans in possible_translations)
                    
                    if incorrect_mistranslated and not correct_translated:
                        errors_detected.append({
                            "English Sentence": en_line,
                            "Incorrect Chinese": zh_line,
                            "Correct Chinese": zh_correct_line,
                            "Polysemous Word": word,
                            "Possible Translations": ", ".join(possible_translations)
                        })
    
    return errors_detected

In [18]:
# File paths before adding salient words
en_file_path_before = "./en-zh.en-filtered.en.subword.test.desubword"
zh_file_path_before = "./zh.translated.desubword"
zh_correct_file_path_before = "./en-zh.zh-filtered.zh.subword.test.desubword"

errors_before = detect_polysemy_errors(en_file_path_before, zh_file_path_before, zh_correct_file_path_before)

pd.DataFrame(errors_before)

,English Sentence,Incorrect Chinese,Correct Chinese,Polysemous Word,Possible Translations
0,"Right now, I am just making my institute in Br...",我们现在对数学教育有个真正的问题。,▁目前我们的数学教育面临着实际的问题。,right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
1,I heard my heart's valves snapping open and cl...,"匿名者:美国福克斯新闻,它引起了我们对于 匿名者名字和自然的关注。",匿名:亲爱的福克斯新闻 很不幸得引起了我们的注意▁所有匿名者的名称和性质 已经被破坏,close,"总结, 使靠拢, 亲密+的, 不通风+的, 没有风+的, 接近+地, 曲终人散, 质地细密的..."
2,A world where you're living at the frontier.,"让我给你们一个概括一下, 什么是无人飞行器。",▁让我总结一下▁“禁捕”保护区的益处,living,"生活+的, 有生命+的, 生计, 住, 忠于生活的, 居住于, 绝对+的, 忍耐, 过着, ..."
3,The question now -- and this is the really int...,"我相信,如果我们继续看护林的话, 应该不会发生, 好好对待我们的社区, 像对待人类一样, 先...","▁我相信这从来不应该发生。 假设我们继续▁以我们认同的方式, 服务我们的社区,▁以人为本,以...",right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
4,The question now -- and this is the really int...,"我相信,如果我们继续看护林的话, 应该不会发生, 好好对待我们的社区, 像对待人类一样, 先...","▁我相信这从来不应该发生。 假设我们继续▁以我们认同的方式, 服务我们的社区,▁以人为本,以...",cut,"稀释, 等级, 不打招呼, 冷落, 凿出的道, 通道, 缩短+的, 剪短, 割, 做, 一份..."
5,"And, in fact, when we did the interview -- I d...","1 加 2 加 3 等于 5, 3 加 5 是 8, 等等.","1 加 2 等于 3 2 加 3 等于 5, 3 加 5 等于 8▁以此类推.",living,"生活+的, 有生命+的, 生计, 住, 忠于生活的, 居住于, 绝对+的, 忍耐, 过着, ..."
6,"Half the challenge is to get access, is to be ...","我的父亲对法律的尊重受到了极大的尊敬, 尽管他被判刑, 但他从没想过错误论文。",我父亲一直被教导要做守法公民▁虽然他受到迫害▁但从没想过办假证件这回事,right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
7,But it's only in the last decade or so that ca...,"但是,你发现的不是人类遗留下的残骸, 比如塞拉姆和露西,每天的。","▁但是你找到的并不是在通常意义上存在的人类, 就像是塞勒姆和露西。",last,"连续, 完结, 延续, 最后阶段, 忍耐, 终点+的, 最後, 过着, 确定性+的, 结束,..."


In [20]:
# File paths after adding salient words
en_file_path_after = "./en-zh.en-filtered-salient.en.subword.test.desubword"
zh_file_path_after = "./zh.translated.desubword"
zh_correct_file_path_after = "./en-zh.zh-filtered.zh.subword.test.desubword"

errors_after = detect_polysemy_errors(en_file_path_after, zh_file_path_after, zh_correct_file_path_after)

pd.DataFrame(errors_before)

,English Sentence,Incorrect Chinese,Correct Chinese,Polysemous Word,Possible Translations
0,"Right now, I am just making my institute in Br...",我们现在对数学教育有个真正的问题。,▁目前我们的数学教育面临着实际的问题。,right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
1,I heard my heart's valves snapping open and cl...,"匿名者:美国福克斯新闻,它引起了我们对于 匿名者名字和自然的关注。",匿名:亲爱的福克斯新闻 很不幸得引起了我们的注意▁所有匿名者的名称和性质 已经被破坏,close,"总结, 使靠拢, 亲密+的, 不通风+的, 没有风+的, 接近+地, 曲终人散, 质地细密的..."
2,A world where you're living at the frontier.,"让我给你们一个概括一下, 什么是无人飞行器。",▁让我总结一下▁“禁捕”保护区的益处,living,"生活+的, 有生命+的, 生计, 住, 忠于生活的, 居住于, 绝对+的, 忍耐, 过着, ..."
3,The question now -- and this is the really int...,"我相信,如果我们继续看护林的话, 应该不会发生, 好好对待我们的社区, 像对待人类一样, 先...","▁我相信这从来不应该发生。 假设我们继续▁以我们认同的方式, 服务我们的社区,▁以人为本,以...",right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
4,The question now -- and this is the really int...,"我相信,如果我们继续看护林的话, 应该不会发生, 好好对待我们的社区, 像对待人类一样, 先...","▁我相信这从来不应该发生。 假设我们继续▁以我们认同的方式, 服务我们的社区,▁以人为本,以...",cut,"稀释, 等级, 不打招呼, 冷落, 凿出的道, 通道, 缩短+的, 剪短, 割, 做, 一份..."
5,"And, in fact, when we did the interview -- I d...","1 加 2 加 3 等于 5, 3 加 5 是 8, 等等.","1 加 2 等于 3 2 加 3 等于 5, 3 加 5 等于 8▁以此类推.",living,"生活+的, 有生命+的, 生计, 住, 忠于生活的, 居住于, 绝对+的, 忍耐, 过着, ..."
6,"Half the challenge is to get access, is to be ...","我的父亲对法律的尊重受到了极大的尊敬, 尽管他被判刑, 但他从没想过错误论文。",我父亲一直被教导要做守法公民▁虽然他受到迫害▁但从没想过办假证件这回事,right,"右手+的, 往右, 补偿, 右边地, 补救, 正直+的, 对, 右倾+的, 正确+的, 扶正..."
7,But it's only in the last decade or so that ca...,"但是,你发现的不是人类遗留下的残骸, 比如塞拉姆和露西,每天的。","▁但是你找到的并不是在通常意义上存在的人类, 就像是塞勒姆和露西。",last,"连续, 完结, 延续, 最后阶段, 忍耐, 终点+的, 最後, 过着, 确定性+的, 结束,..."
